# Fine-Tuned DistilBERT Classifier — Colab Pro (A100 GPU)

**NSF NRT Challenge 1 | Member 1 — Tony Nguyen**

Fine-tunes `distilbert-base-uncased` on pseudo-labeled substance-abuse posts.

> **Before running:** `Runtime → Change runtime type → A100 GPU → Save`

Estimated training time: ~10–15 min on A100.

## Cell 1 — Detect Hardware & Install Dependencies

In [1]:
import os, torch

# Detect GPU
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected — switch runtime to A100 GPU for best performance.')

# Upgrade to latest compatible versions — fixes peft/accelerate conflicts
!pip install -q -U transformers datasets accelerate scikit-learn
# Must restart runtime after install for changes to take effect
import importlib, accelerate, transformers
print(f'transformers: {transformers.__version__}')
print(f'accelerate  : {accelerate.__version__}')
print('Install complete — proceed to Cell 2.')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
transformers: 5.5.0
accelerate  : 1.13.0
Install complete — proceed to Cell 2.


## Cell 2 — Mount Google Drive

In [2]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

# Edit to match your Drive folder
DRIVE_DIR = '/content/drive/MyDrive/NSF_NRT_SubstanceAbuse'
os.makedirs(f'{DRIVE_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models/finetuned_bert', exist_ok=True)
print('Drive folder ready:', DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive folder ready: /content/drive/MyDrive/NSF_NRT_SubstanceAbuse


## Cell 3 — Upload Data Files

Upload from your local project:
- `data/processed/ensemble_results.csv`
- `data/processed/posts_preprocessed.csv`

Use **Option A** (file picker) or **Option B** (already on Drive).

In [12]:
# Option A: upload from local machine
from google.colab import files as colab_files
print('Select ensemble_results.csv and posts_preprocessed.csv')
uploaded = colab_files.upload()
DATA_DIR = f'{DRIVE_DIR}/data/processed'
for fname in uploaded:
    dest = f'{DATA_DIR}/{fname}'
    shutil.copy(fname, dest)
    print(f'  Saved {fname} -> {dest}')

Select ensemble_results.csv and posts_preprocessed.csv


Saving ensemble_results.csv to ensemble_results.csv
Saving posts_preprocessed.csv to posts_preprocessed (1).csv
  Saved ensemble_results.csv -> /content/drive/MyDrive/NSF_NRT_SubstanceAbuse/data/processed/ensemble_results.csv
  Saved posts_preprocessed (1).csv -> /content/drive/MyDrive/NSF_NRT_SubstanceAbuse/data/processed/posts_preprocessed (1).csv


In [ ]:
# Option B: project already synced to Drive — edit SRC and uncomment
# SRC = '/content/drive/MyDrive/Substance_Abuse_Detection_AI/data/processed'
# DATA_DIR = f'{DRIVE_DIR}/data/processed'
# for f in ['ensemble_results.csv', 'posts_preprocessed.csv']:
#     shutil.copy(f'{SRC}/{f}', f'{DATA_DIR}/{f}')
#     print(f'Copied {f}')

## Cell 4 — Load & Prepare Data

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

LABEL2ID = {'low': 0, 'medium': 1, 'high': 2}
ID2LABEL  = {0: 'low', 1: 'medium', 2: 'high'}
SEED = 42

# Define DATA_DIR here to fix NameError if Cell 3 was skipped
DATA_DIR = f'{DRIVE_DIR}/data/processed'

ens = pd.read_csv(f'{DATA_DIR}/ensemble_results.csv')
pre = pd.read_csv(f'{DATA_DIR}/posts_preprocessed.csv')

text_col = next(
    (c for c in ('processed_text', 'cleaned_text', 'original_text') if c in pre.columns),
    None
)
print(f'Text column: {text_col}')
print(f'Ensemble: {len(ens):,}  |  Preprocessed: {len(pre):,}')

# Row-position alignment (handles NaN post_id in preprocessed CSV)
n  = min(len(ens), len(pre))
df = pd.DataFrame({
    'post_id':          range(n),
    'final_risk_level': ens['final_risk_level'].values[:n],
    'text':             pre[text_col].values[:n],
})
df = df.dropna(subset=['text', 'final_risk_level'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 10]
df = df[df['final_risk_level'].isin(LABEL2ID)].reset_index(drop=True)
df['label'] = df['final_risk_level'].map(LABEL2ID)
print(f'Usable posts: {len(df):,}')
print(df['final_risk_level'].value_counts())

Text column: processed_text
Ensemble: 26,894  |  Preprocessed: 26,894
Usable posts: 26,859
final_risk_level
medium    16407
low        8924
high       1528
Name: count, dtype: int64


## Cell 5 — Tokenise

In [4]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH  = 256
tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, val_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=SEED
)
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}')

def make_hf_dataset(frame):
    ds = Dataset.from_pandas(frame[['post_id', 'text', 'label']].reset_index(drop=True))
    return ds.map(
        lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH, padding=False),
        batched=True, remove_columns=['text']
    )

train_ds = make_hf_dataset(train_df)
val_ds   = make_hf_dataset(val_df)
print('Tokenisation complete.')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 21,487  |  Val: 5,372


Map:   0%|          | 0/21487 [00:00<?, ? examples/s]

Map:   0%|          | 0/5372 [00:00<?, ? examples/s]

Tokenisation complete.


## Cell 6 — Fine-Tune on A100 GPU

Effective batch size: 16 per device. `bf16=True` is supported natively on A100 for speed. Estimated time: **~10–15 min**.

In [8]:
import json
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import torch

MODEL_DIR = f'{DRIVE_DIR}/models/finetuned_bert'

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_high':  f1_score(labels, preds, labels=[2], average='micro', zero_division=0),
    }

model    = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# Calculate warmup_steps explicitly (warmup_ratio deprecated in transformers v5)
steps_per_epoch = len(train_ds) // 16
warmup_steps    = int(steps_per_epoch * 3 * 0.06)  # 6% of total steps
print(f'steps_per_epoch={steps_per_epoch}  warmup_steps={warmup_steps}')

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    bf16=use_bf16,
    fp16=use_fp16,
    seed=42,
    report_to='none',
    logging_steps=50,
    dataloader_num_workers=2,
)

# 'processing_class' replaced 'tokenizer' in transformers v5
import transformers as _tf
_trainer_kwargs = dict(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
if int(_tf.__version__.split('.')[0]) >= 5:
    _trainer_kwargs['processing_class'] = tokenizer
else:
    _trainer_kwargs['tokenizer'] = tokenizer
trainer = Trainer(**_trainer_kwargs)

print(f'Training on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'bf16={use_bf16}  fp16={use_fp16}')
trainer.train()

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
with open(f'{MODEL_DIR}/label_map.json', 'w') as fp:
    json.dump({'id2label': ID2LABEL, 'label2id': LABEL2ID}, fp, indent=2)

print(f'Model saved -> {MODEL_DIR}')

# Remove notebook progress callback to avoid evaluation bug by checking class name dynamically
callbacks_to_remove = [c for c in trainer.callback_handler.callbacks if c.__class__.__name__ == "NotebookProgressCallback"]
for c in callbacks_to_remove:
    trainer.remove_callback(c)

for k, v in trainer.evaluate().items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


steps_per_epoch=1342  warmup_steps=241
Training on: NVIDIA A100-SXM4-40GB
bf16=True  fp16=False


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 High
1,0.721828,0.767429,0.644639,0.373036,0.000000
2,0.702002,0.765971,0.643150,0.477923,0.188172
3,0.661380,0.766405,0.656366,0.487608,0.221675


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved -> /content/drive/MyDrive/NSF_NRT_SubstanceAbuse/models/finetuned_bert
  eval_loss: 0.7664
  eval_accuracy: 0.6564
  eval_f1_macro: 0.4876
  eval_f1_high: 0.2217
  eval_runtime: 1.7839
  eval_samples_per_second: 3011.3330
  eval_steps_per_second: 94.1740
  epoch: 3.0000


## Cell 7 — Full-Dataset Inference

In [9]:
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1
clf = hf_pipeline(
    'text-classification',
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=device,
    truncation=True,
    max_length=MAX_LENGTH,
    batch_size=64,
)

texts = df['text'].fillna('').astype(str).tolist()
print(f'Running inference on {len(texts):,} posts ...')
raw = clf(texts)

label_map = {'LABEL_0': 'low', 'LABEL_1': 'medium', 'LABEL_2': 'high'}
OUT_CSV   = f'{DATA_DIR}/finetuned_results.csv'
results   = pd.DataFrame({
    'post_id':    df['post_id'].values,
    'risk_level': [label_map.get(r['label'], r['label'].lower()) for r in raw],
    'confidence': [round(r['score'], 4) for r in raw],
})
results.to_csv(OUT_CSV, index=False)
print(f'Saved {len(results):,} predictions -> {OUT_CSV}')
print(results['risk_level'].value_counts())

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Running inference on 26,859 posts ...
Saved 26,859 predictions -> /content/drive/MyDrive/NSF_NRT_SubstanceAbuse/data/processed/finetuned_results.csv
risk_level
medium    20068
low        6299
high        492
Name: count, dtype: int64


## Cell 8 — Download Results

In [10]:
from google.colab import files as colab_files
colab_files.download(OUT_CSV)
print('Downloaded finetuned_results.csv')
print('Place it in your project at: data/processed/finetuned_results.csv')
print(f'Model checkpoint is in Drive at: {MODEL_DIR}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded finetuned_results.csv
Place it in your project at: data/processed/finetuned_results.csv
Model checkpoint is in Drive at: /content/drive/MyDrive/NSF_NRT_SubstanceAbuse/models/finetuned_bert
